In [ ]:
"""
This script implements a machine learning pipeline for classifying handwritten digits from the MNIST dataset using a HistGradientBoostingClassifier. The pipeline includes data loading, image augmentation, model training, evaluation, and result visualization.

The code performs the following steps:

1. **Load MNIST Dataset**: The MNIST dataset is fetched from OpenML, and the images are reshaped into a 28x28 format.

2. **Image Augmentation**: A function `augment_image` is defined to perform basic augmentations on the images,
     including horizontal and vertical flips, as well as a 90-degree clockwise rotation.

3. **Generate Augmented Dataset**: The script iterates through the original dataset, appending each original image
     and its augmentations to new lists. It also prints progress updates every 10,000 images processed.

4. **Data Preparation**: The augmented images and labels are converted to NumPy arrays and reshaped for machine learning. The dataset is then split into training, validation, and test sets using stratified sampling.

5. **Hyperparameter Tuning**: A parameter grid is defined for the HistGradientBoostingClassifier, and GridSearchCV
     is used to find the best hyperparameters based on accuracy.

6. **Model Training**: The best model is trained on the training dataset, and the best parameters and accuracy are
    printed.

7. **Model Saving**: The trained model is saved to a file for future use.

8. **Learning Curve Visualization**: The learning curve is plotted to visualize the mean squared error (MSE) for
     both training and validation sets as a function of the training size.

9. **Model Evaluation**: The model is evaluated on the training and test sets. Mean Squared Error (MSE) and Mean
     Absolute Error (MAE) are calculated and printed. A confusion matrix is displayed, and a classification report
     is generated.

10. **Feature Importance Visualization**: The script calculates and visualizes the importance of the top 20 pixel
      features used by the model.

11. **Results Packaging**: The model and visualizations are zipped into a single file for download.

12. **Shutdown Runtime**: The script includes a command to shut down the Colab runtime after the file download is
      triggered.

Dependencies:
- numpy
- matplotlib
- scikit-learn
- joblib
- opencv-python
- google.colab

Usage:
Run the script in a Python environment with the required libraries installed. The output will include model
training results, visualizations, and a downloadable zip file containing the trained model and plots.
"""


import numpy as np
import matplotlib.pyplot as plt
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split, learning_curve
from sklearn.metrics import (
    confusion_matrix,
    ConfusionMatrixDisplay,
    mean_squared_error,
    mean_absolute_error,
    classification_report
)
import joblib
from google.colab import files
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.model_selection import GridSearchCV



# Load and reshape data
mnist = fetch_openml("mnist_784", version=1, as_frame=False)
X, y = mnist["data"], mnist["target"].astype(np.uint8)
X = X.reshape(-1, 28, 28)

# Augment each image with H-flip, V-flip, 90° rotation
import cv2

def augment_image(img):
    img_h = cv2.flip(img, 1)
    img_v = cv2.flip(img, 0)
    img_rot = cv2.rotate(img, cv2.ROTATE_90_CLOCKWISE)
    return [img_h, img_v, img_rot]

aug_images = []
aug_labels = []

for i in range(len(X)):
    img = X[i]
    label = y[i]
    aug_images.append(img)
    aug_labels.append(label)
    for aug in augment_image(img):
        aug_images.append(aug)
        aug_labels.append(label)
    if (i + 1) % 10000 == 0:
        print(f"Processed {i+1} / {len(X)}")

# Final dataset
aug_images = np.array(aug_images).reshape(len(aug_images), -1)
aug_labels = np.array(aug_labels)

# Stratified split
X_temp, X_test, y_temp, y_test = train_test_split(
    aug_images, aug_labels, test_size=0.2, stratify=aug_labels, random_state=42
)
X_train, X_val, y_train, y_val = train_test_split(
    X_temp, y_temp, test_size=0.25, stratify=y_temp, random_state=42
)

print(f"Train size: {len(X_train)}, Val size: {len(X_val)}, Test size: {len(X_test)}")



param_grid = {
    "learning_rate": [0.05, 0.1],
    "max_iter": [100, 200],
    "max_leaf_nodes": [31, 63],
    "min_samples_leaf": [20, 30],
    "l2_regularization": [0.0, 1.0]
}

# Train HistGradientBoostingClassifier
clf = HistGradientBoostingClassifier(random_state=42)
grid = GridSearchCV(clf, param_grid, cv=3, scoring="accuracy", n_jobs=-1, verbose=1)
grid.fit(X_train, y_train)

print("Best Parameters:", grid.best_params_)
print("Best Accuracy:", grid.best_score_)

# 2. Save model
joblib.dump(clf, "hgb_classifier.joblib")
print("✅ Model saved to 'hgb_classifier.joblib'")

# Learning Curve (use MSE as scoring)
train_sizes, train_scores, val_scores = learning_curve(
    clf, X_train, y_train, train_sizes=np.linspace(0.1, 1.0, 5),
    cv=3, scoring='neg_mean_squared_error', n_jobs=-1
)
train_errors = -train_scores.mean(axis=1)
val_errors = -val_scores.mean(axis=1)

plt.figure(figsize=(8, 5))
plt.plot(train_sizes, train_errors, label="Train MSE")
plt.plot(train_sizes, val_errors, label="Validation MSE")
plt.xlabel("Training Set Size")
plt.ylabel("Mean Squared Error")
plt.title("Learning Curve (HistGradientBoostingClassifier)")
plt.legend()
plt.grid(True)
plt.savefig("hgb_learning_curve.png")
plt.show()

# Predict
y_train_pred = clf.predict(X_train)
y_test_pred = clf.predict(X_test)

# MSE and MAE
mse_train = mean_squared_error(y_train, y_train_pred)
mse_test = mean_squared_error(y_test, y_test_pred)
mae_train = mean_absolute_error(y_train, y_train_pred)
mae_test = mean_absolute_error(y_test, y_test_pred)

print(f"\nTrain MSE: {mse_train:.4f}, MAE: {mae_train:.4f}")
print(f"Test  MSE: {mse_test:.4f}, MAE: {mae_test:.4f}")

# Confusion Matrix
cm = confusion_matrix(y_test, y_test_pred)
disp = ConfusionMatrixDisplay(cm)
disp.plot(cmap='Blues')
plt.title("Test Confusion Matrix")
plt.show()

# Classification Report
print("\nClassification Report (Test Set):\n")
print(classification_report(y_test, y_test_pred))

# Feature Importances
importances = clf.feature_importances_
top_features = np.argsort(importances)[-20:][::-1]  # Top 20
top_values = importances[top_features]

# Feature Names (pixels)
feature_names = [f"pixel_{i}" for i in top_features]

plt.figure(figsize=(10, 6))
plt.barh(feature_names, top_values)
plt.gca().invert_yaxis()
plt.title("Top 20 Pixel Feature Importances")
plt.xlabel("Importance")
plt.grid(True)
plt.tight_layout()
plt.savefig("hgb_feature_importance.png")
plt.show()

# Print full importances (optional)
for idx in top_features:
    print(f"Feature pixel_{idx}: Importance = {importances[idx]:.6f}")

# STEP 7: Zip Results
import zipfile

with zipfile.ZipFile("hgb_outputs.zip", "w") as zipf:
    zipf.write("hgb_classifier.joblib")
    zipf.write("hgb_learning_curve.png")
    zipf.write("hgb_feature_importance.png")

files.download("hgb_outputs.zip")


# STEP 8: Shut down Colab runtime
import IPython
print("✅ Training complete. Runtime will shut down after file is downloaded.")
IPython.display.display(IPython.display.Javascript('''
  async function shutdown() {
    await google.colab.kernel.invokeFunction('shutdown');
  }
  shutdown();
'''))